## Basics

In [1]:
#import relevant libraries
import os
from scipy import stats
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap
import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import osar 

#NOTE: SUPPRESSES WARNINGS!

import warnings
warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

print(osar.__version__)

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 23.74it/s]


Numba compilation complete!
0.23.7


In [2]:
officecomp = "C:\\Users\\Star\\"
labcomp = "C:\\Users\\User\\"
computer2 = "C:\\Users\\lnico\\"
homecomp = "D:\\"
specifiedpath = homecomp

filedirectory_OSAR = "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\2025collection\\"
filedirectory_Falling = "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\Compilation with delta\\2025deltagcollection\\"
filesavedir = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Totalosarfalling\\images\\"
filedate = "20251014"

newfile2 = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\Compilation with delta\\2025fallingtoosarcomp\\"
files2 = os.listdir(newfile2)

In [3]:
#for fonts only
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_dirs = [specifiedpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Programs\\"]  # The path to the custom font file.
font_files = font_manager.findSystemFonts(fontpaths=font_dirs)


for font_file in font_files:
    font_manager.fontManager.addfont(font_file)
    prop = font_manager.FontProperties(fname=font_file)

availablefonts = [f.name for f in matplotlib.font_manager.fontManager.ttflist]
plt.rcParams["font.family"] = "Inter"

## Function

In [4]:
def reading_OSAR_compiled_files(df, responder):
    df['genotypeandresponder'] = df['MBON'] + "_" + responder
    
    return df

def forestplot_multiplot(sorteddf, df1, parameter):
    lst = []
    names =[]
    for n in sorteddf['MBON']:
        tempdf1 = df1[(df1['MBON'] == n)].reset_index(drop=True)
        dbfeature = dabest.load(tempdf1, idx=("Sibling", "Offspring"),x="status", y= parameter)
        lst.append(dbfeature)
        names.append(n)
    return lst, names



In [5]:
newfile2 = specifiedpath + filedirectory_Falling  # from 2025deltagcollection
files2 = os.listdir(newfile2)

## Single files for Falling

In [ ]:

totalfile = pd.DataFrame()

#considered to be at half light intensity in OSAR as well

for n in files2:
    openfile = pd.read_csv(newfile2 + n)
    nameofmbon = n.split(" ")[0] 
    openfile['responder'] = n.split(" x ")[1].split("_")[0]
    totalfile = pd.concat([totalfile, openfile], axis = 0).reset_index(drop=True)

lstofallvariables = []
for n in totalfile.columns.to_list()[1:-1]:
    lstofallvariables.append(n.split("_")[0])
lstnamings= list(set(lstofallvariables))

totalfile['genotypeandresponder'] = totalfile["MBON"] + "_" + totalfile['responder']   #has bootstraps

#only delta-g
onlymeans = totalfile.loc[:, ~totalfile.columns.str.endswith('_bootstrap')].drop_duplicates().reset_index(drop=True)

mbononly=onlymeans[~onlymeans['MBON'].isin(['R58', 'Th-Gal4'])]

# to generate excel file only
fallingexcel = mbononly.copy()
fallingexcel.columns = fallingexcel.columns.str.replace('_deltag', '_Δg')
fallingexcel = fallingexcel.rename(columns={'speed_Δg': 'Speed_Δg', 
                                            'bspeed_Δg': 'Bout speed_Δg', 
                                            'pausepos_Δg': 'Pause position_Δg', 
                                            'bout_Δg': 'Number of Bouts_Δg', 
                                            'meanbout_Δg': "Duration of bouts_Δg",
                      'straightindex_Δg': 'Straightness Index_Δg', 
                      'height_Δg': 'Max height climbed_Δg', 
                      'maxvelocity_Δg': 'Max velocity_Δg', 
                      "fallnumber_meandiff": "Fall number_ΔΔ"}).reset_index(drop=True)
fallingexcel.to_excel(specifiedpath+ "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\Falling.xlsx", index = False)

## Single files for OSAR from already generated compiled sheet

In [ ]:
responder = "Chrimson2"
#from dabest.forest_plot import forest_plot
compiled = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\"+ filedate + "_" + responder + "_totalcompilation.csv"
dfcompiled= pd.read_csv(compiled, index_col=0)
dfcompiled.replace([np.inf, -np.inf], np.nan, inplace=True)
dfcompiled.rename(columns={'driver': 'MBON'}, inplace = True)
dfcompiled=dfcompiled[~dfcompiled['MBON'].isin(['R58', 'Th-Gal4'])]

In [ ]:
# Define the four conditions
conditions = {
    'Eighth': dfcompiled['light_intensity'] == 'Eighth',
    'Quarter': dfcompiled['light_intensity'] == 'Quarter',
    'Half': dfcompiled['light_intensity'] == 'Half',
    'Full': dfcompiled['light_intensity'] == 'Full',
    'Half_and_Full': (dfcompiled['light_intensity'] == 'Half') | (dfcompiled['light_intensity'] == 'Full')
}

for condition_name, condition_filter in conditions.items():
    # Filter data based on condition
    df1 = dfcompiled[condition_filter].loc[:,['MBON','light_intensity','status','pi_smoothed_Pattern 01', 
                                               'log2_speed_ratio_Pattern 01','log2_pace_ratio_Pattern 01',
                                               'light_attraction_index_Pattern 01']].reset_index(drop=True)
    
    sorteddf = pd.DataFrame()
    
    for n in df1['MBON'].unique().tolist():
        dbfeature = dabest.load(df1[(df1['MBON'] == n)].reset_index(drop=True), idx=("Sibling", "Offspring"),x="status", y="pi_smoothed_Pattern 01")
        tempdf = pd.DataFrame()
        tempdf['MBON'] = [n]
        tempdf['difference']= [float(dbfeature.hedges_g.statistical_tests.difference)]
        sorteddf = pd.concat([sorteddf, tempdf], axis =0)
    
    sorteddf = sorteddf.sort_values(by=['difference']).reset_index(drop=True)
    
    lstpi, namepi = forestplot_multiplot(sorteddf, df1, "pi_smoothed_Pattern 01")
    lstsp, namesp = forestplot_multiplot(sorteddf, df1, "log2_speed_ratio_Pattern 01")
    lstpc, namepc = forestplot_multiplot(sorteddf, df1, "log2_pace_ratio_Pattern 01")
    lstlei, namelei = forestplot_multiplot(sorteddf, df1, "light_attraction_index_Pattern 01")
    
    # Generate excel files
    parameters= ["pi_smoothed_Pattern 01", "log2_speed_ratio_Pattern 01", "log2_pace_ratio_Pattern 01", "light_attraction_index_Pattern 01"]
    parameters_rename= ["Δ PI", "Δ Log2_Speed_ratio", "Δ Log2_Bout Speed_ratio", "Δ Light Attraction Index"]
    
    result_df = pd.DataFrame({'MBON': sorteddf['MBON'].unique()})
    
    for parameter, newname in zip(parameters, parameters_rename):
        differences = []
        
        for n in result_df['MBON']:
            tempdf1 = df1[(df1['MBON'] == n)].reset_index(drop=True)
            dbfeature = dabest.load(tempdf1, idx=("Sibling", "Offspring"), x="status", y=parameter)
            differences.append(dbfeature.hedges_g.statistical_tests.difference[0])
        
        # Add as new column
        result_df[newname] = differences
    
    # Save with condition name in filename
    result_df.to_excel(specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\OSAR_" + responder + "-" + condition_name + ".xlsx", index=False)
    
    print(f"Saved: OSAR_{responder}_{condition_name}.xlsx")

Saved: OSAR_Chrimson2_Eighth.xlsx
Saved: OSAR_Chrimson2_Quarter.xlsx
Saved: OSAR_Chrimson2_Half.xlsx
Saved: OSAR_Chrimson2_Full.xlsx
Saved: OSAR_Chrimson2_Half_and_Full.xlsx


## Log2Speedratio integration for OSAR-Falling

In [8]:
# To run if need new files to generate for compiled work
# falling stuff
from collections import Counter
counts = Counter([item.split(' ')[0] for item in files2])
matchingsets = [value for value, count in counts.items() if count > 1]  #only if you have both ACR and Chrimson data
matchinglst = [v for v in files2 if any(short_name in v for short_name in matchingsets)]
dffall_ACR = pd.DataFrame()
dffall_Chrimson2 = pd.DataFrame()

for f in matchinglst:
    if "Chrimson2" in f:
        temp = pd.read_csv(newfile2 + f, index_col=0)
        temp = temp.loc[:, ~temp.columns.str.endswith('_bootstrap')].drop_duplicates()
        dffall_Chrimson2 = pd.concat([dffall_Chrimson2, temp], axis = 0)
    if "ACR" in f:
        temp = pd.read_csv(newfile2 + f, index_col=0)
        temp = temp.loc[:, ~temp.columns.str.endswith('_bootstrap')].drop_duplicates()
        dffall_ACR = pd.concat([dffall_ACR, temp], axis = 0)
        

dffall_Chrimson2 = dffall_Chrimson2.reset_index(drop=False)
dffall_Chrimson2['genotypeandresponder'] = dffall_Chrimson2['MBON'] + '_Chrimson2'
dffall_ACR = dffall_ACR.reset_index(drop=False)
dffall_ACR['genotypeandresponder'] = dffall_ACR['MBON'] + '_ACR'

##OSAR
# Define responders and conditions
responders = ['Chrimson2', 'ACR']
conditions = ['Eighth', 'Quarter', 'Half', 'Full', 'Half_and_Full']

# Dictionary to store merged dataframes
merged_dfs = {}

for condition in conditions:
    condition_dfs = []
    
    for responder in responders:
        file_path = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\OSAR_" + responder + "-" + condition + ".xlsx"

        df_osarlightintensities = pd.read_excel(file_path)
        df_osarlightintensities = df_osarlightintensities[['MBON', 'Δ PI', 'Δ Log2_Speed_ratio', 'Δ Log2_Bout Speed_ratio']]
        
        # Optionally add a responder column to track which responder the data came from
        df_osarlightintensities['Responder'] = responder
        df_osarlightintensities['genotypeandresponder'] = df_osarlightintensities['MBON'] + "_" + responder
        condition_dfs.append(df_osarlightintensities)
    
    # Merge all responders for this condition
    merged_dfs[condition] = pd.concat(condition_dfs, axis=0, ignore_index=True)

# Access individual merged dataframes
df_osareighth = merged_dfs['Eighth']
df_osarquarter = merged_dfs['Quarter']
df_osarhalf = merged_dfs['Half']
df_osarfull = merged_dfs['Full']
df_osarhalf_and_full = merged_dfs['Half_and_Full']


#together

dfosarintensities = [df_osareighth,df_osarquarter, df_osarhalf, df_osarfull, df_osarhalf_and_full]
dffalltotal= pd.concat([dffall_ACR, dffall_Chrimson2], axis = 0).reset_index(drop=True)
dffallingtotal = dffalltotal.loc[:,['genotypeandresponder', 'bspeed_deltag', 'speed_deltag']]

for m, condition_resp in zip(dfosarintensities, conditions):
    dfmix = m.merge(dffallingtotal, on="genotypeandresponder", how= "inner")
    #dfmix = dfmix.drop(['responder'], axis = 1)
    dfmix = dfmix.sort_values(by = ["genotypeandresponder"])

    osarmatchdf = dfmix[~dfmix['MBON'].isin(['R58', 'Th-Gal4'])]

    #to generate excel file only
    osarexcel = osarmatchdf.copy()
    osarexcel = osarexcel.rename(columns={'Δ PI': 'Δ PI_OSAR', 'Δ Log2 Speed Ratio': 'Δ Log2 Speed Ratio_OSAR', 'Δ Log2 Pace Ratio': 'Δ Log2 Bout Speed Ratio_OSAR'
                                , 'bspeed_deltag': 'Δ Log2 Bout Speed Ratio_Falling', 'speed_deltag': 'Δ Log2 Speed Ratio_Falling'}).reset_index(drop=True)
    osarexcel.to_excel(specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\OSAR and Falling_delta Log2SR and delta Log2BSR comparison-" + condition_resp + ".xlsx", index = False)

## OSAR-Falling Combination

In [10]:
#pre processed fallin
root = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\"
filefalling = root + "Falling.xlsx"
df_falling = pd.read_excel(filefalling)

cols = df_falling.columns.tolist()
cols[1:-2] = [col + '_Falling' for col in cols[1:-2]]
df_falling.columns = cols

df_falling = df_falling.drop(columns= ["MBON"])

lightintensities = ['Eighth', 'Quarter', 'Half', 'Full', 'Half_and_Full']
root = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\"

for light in lightintensities:
    filepathACR = root + "OSAR_ACR-"+ light +".xlsx"
    filepathCR2 = root + "OSAR_Chrimson2-" + light + ".xlsx"
    filepathlogratio = root + "OSAR and Falling_delta Log2SR and delta Log2BSR comparison-" + light + ".xlsx"  #File generated from 2025fallingtoosarcomp

    dffileACR= reading_OSAR_compiled_files(pd.read_excel(filepathACR), "ACR")
    dffileCR2= reading_OSAR_compiled_files(pd.read_excel(filepathCR2),'Chrimson2')
    dfosartotal= pd.concat([dffileACR, dffileCR2], axis = 0).reset_index(drop=True)
    
    dffilelogratio = pd.read_excel(filepathlogratio)    
    dfosartotal = dfosartotal.rename(columns={'Δ PI': 'PI_Δg_OSAR', "Δ Log2_Speed_ratio": "Log2 Speed ratio_Δg_OSAR",
                                            "Δ Log2_Bout Speed_ratio": "Log2 Bout Speed ratio_Δg_OSAR", "Δ Light Attraction Index": "Light Attraction Index_Δg_OSAR"
                                            })

    dflogratiofalling = dffilelogratio.iloc[:,-3:].rename(columns={"Δ Log2 Speed Ratio_Falling": "Log2 Speed ratio_Δg_Falling",
                                            "Δ Log2 Bout Speed Ratio_Falling": "Log2 Bout Speed ratio_Δg_Falling"
                                            })
    dfmix = pd.DataFrame()
    dfmix = df_falling.merge(dfosartotal, on="genotypeandresponder", how="inner") \
                  .merge(dflogratiofalling, on="genotypeandresponder", how="inner")
    dfmix = dfmix.sort_values(by = ["genotypeandresponder"])

    totalmetriccomparison = dfmix[~dfmix['MBON'].isin(['R58', 'Th-Gal4'])]

    totalmetriccomparison.to_excel(specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\Totalcomparisonofallmetrics-" + light +".xlsx", index = False)
    print(f"Saved: {light}.xlsx")

Saved: Eighth.xlsx
Saved: Quarter.xlsx
Saved: Half.xlsx
Saved: Full.xlsx
Saved: Half_and_Full.xlsx
